Copyright Matlantis Corp. as contributors to Matlantis contrib project

# NPzT Equilibration MD of Cu(111)/Water Interface

Equilibrate the Cu(111)/water interface structure created in the modeling step using NPzT ensemble (pressure control in z-direction only) MD simulation with the PFP potential.

The main steps in this notebook are:
1. **Load structure:** Load the interface structure created in Notebook 01 and fix the bottom Cu layers.
2. **Run NPzT MD:** Perform 100 ps (100,000 steps) of MD simulation at 375 K.
3. **Save results:** Save the equilibrated structure to file.

## Step 1: Import Libraries and PFP Settings

Import the required ASE modules and PFP Calculator, and configure the calculation settings.

In [ ]:
from pathlib import Path
import numpy as np
from time import perf_counter

# ASE
from ase.io import read, write
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from ase.md.npt import NPT
from ase.md import MDLogger
from ase.constraints import FixAtoms
from ase import units

# PFP calculator
from pfcc_extras.visualize.view import view_ngl
from pfp_api_client.pfp.estimator import Estimator, EstimatorCalcMode, EstimatorMethodType
from pfp_api_client.pfp.calculators.ase_calculator import ASECalculator

estimator = Estimator(
    model_version="v8.0.0",
    method_type=EstimatorMethodType.PFVM_D3_PFVM,
    calc_mode=EstimatorCalcMode.PBE_PLUS_D3,
)
calculator = ASECalculator(estimator)

out_dir = Path('output/02_md_equilibrium')
out_dir.mkdir(exist_ok=True, parents=True)

## Step 2: Load Structure

Load the optimized interface structure (`cu_water_interface_opt.cif`) created in Notebook 01.

**note**: If the file is not found in `output/01_modeling/`, it will be loaded from `assets/01_modeling/` to allow running this notebook independently.

In [ ]:
import os

# Load from output, fall back to assets if not found
inp_file = "./output/01_modeling/cu_water_interface_opt.cif"
if not os.path.exists(inp_file):
    inp_file = "./assets/01_modeling/cu_water_interface_opt.cif"
    print(f"File not found in output, loading from assets: {inp_file}")

atoms = read(inp_file)
atoms.calc = calculator

view_ngl(atoms, representations=["ball+stick"], w=400, h=300)

## Step 3: Fix Bottom Cu Slab Layers

Fix the atoms in the bottom two layers of the Cu slab (z < 4.0 Å). This keeps the slab base as a fixed reference, allowing more accurate reproduction of the interface behavior.

In [ ]:
# Fix the bottom 1st and 2nd Cu layers

z_fix_threshold = 4.0
constraint = FixAtoms(mask=atoms.positions[:, 2] < z_fix_threshold)
atoms.set_constraint(constraint)
constraint

## Step 4: Run NPzT MD Simulation

Run an MD simulation in the NPzT (pressure control in z-direction only) ensemble.

| Parameter | Value | Description |
|:---|:---|:---|
| Ensemble | NPzT | Cell size varies in z-direction only (`mask=[0,0,1]`) |
| Temperature | 375 K | |
| Pressure | 1.0 bar | |
| Time step | 1.0 fs | |
| Number of steps | 100,000 (= 100 ps) | |
| Log interval | 1,000 steps | |
| `ttime` | 20.0 fs | Thermostat time constant |
| `pfactor` | 2×10⁵ GPa·fs² | Barostat parameter |

In [ ]:
%%time

# input parameters
time_step_fs = 1.0      # fs
temperature_k = 375.0   # K
pressure_bar = 1.0      # bar
num_md_steps = 100_000  # 100ps
num_interval = 1000

ttime_fs = 20.0     # Time constant [fs]
pfactor_gpa = 2e5   # Barostat parameter [GPa]

output_stem = out_dir / 'mdtraj'
log_filename = str(output_stem) + '.log'
traj_filename = str(output_stem) + '.traj'
xyz_filename = str(output_stem) + '.xyz'
cell_log_filename = str(output_stem) + '_cell.log'
print('log_filename  =', log_filename)
print('traj_filename =', traj_filename)
print('xyz_filename  =', xyz_filename)
print('cell_log_filename =', cell_log_filename)

# set the momenta corresponding to the target temperature
MaxwellBoltzmannDistribution(atoms, temperature_K=temperature_k, force_temp=True)
Stationary(atoms)

mask_matrix = np.diag([0, 0, 1])  # only z-axis cell scaling

dyn = NPT(atoms,
          time_step_fs * units.fs,
          temperature_K=temperature_k,
          externalstress=pressure_bar * units.bar,
          ttime=ttime_fs * units.fs,
          mask=mask_matrix,
          pfactor=pfactor_gpa * units.GPa * (units.fs**2),
          logfile=log_filename,
          trajectory=traj_filename,
          loginterval=num_interval)

# Print statements
def print_dyn():
    imd = dyn.get_number_of_steps()
    etot = atoms.get_total_energy()
    temp_K = atoms.get_temperature()
    volume = atoms.get_volume()
    stress = atoms.get_stress(include_ideal_gas=True) / units.GPa
    stress_ave = stress[:3].mean()
    elapsed_time = perf_counter() - start_time
    print(f"  {imd: >3}   {etot:.3f}    {temp_K:.2f}  {volume:.2f}  {stress_ave:.2f}  {stress[0]:.2f}  {stress[1]:.2f}  {stress[2]:.2f}  {stress[3]:.2f}  {stress[4]:.2f}  {stress[5]:.2f}    {elapsed_time:.3f}")

# Open log file and write header
log_file = open(cell_log_filename, 'w')
log_file.write(f"{'Time[ps]':>12s} {'Lx[A]':>12s} {'Ly[A]':>12s} {'Lz[A]':>12s} {'Volume[A^3]':>15s}\n")

# Custom function to record cell information
def log_cell_info(a=atoms):
    """Record simulation cell dimensions and volume to file"""

    # Cell information
    cell = a.get_cell()
    lx = cell[0, 0]
    ly = cell[1, 1]
    lz = cell[2, 2]
    
    # Get volume
    volume = a.get_volume()
    
    # Get time (convert to ps)
    time_ps = dyn.get_time() / (1000 * units.fs)
    
    # Format and write to file
    log_file.write(f"{time_ps:12.4f} {lx:12.6f} {ly:12.6f} {lz:12.6f} {volume:15.6f}\n")
    
    # Flush buffer for immediate write
    log_file.flush()    

dyn.attach(print_dyn, interval=num_interval)
dyn.attach(log_cell_info, interval=num_interval)
dyn.attach(MDLogger(dyn, atoms, log_filename, header=True, stress=True, peratom=True, mode="w"), interval=num_interval)

# Simulation
try:
    start_time = perf_counter()
    print(f"    imd     Etot(eV)    T(K)    volume   stress(mean,xx,yy,zz,yz,xz,xy)(GPa)  elapsed_time(sec)")
    dyn.run(num_md_steps)
finally:
    log_file.close()

## Step 5: Save Equilibrated Structure

Save the structure after MD simulation in extxyz format. This structure will be used as the initial structure for the next Steered MD step.

In [ ]:
write(str(out_dir / 'mdtraj_eq.xyz'), atoms, format='extxyz')

## Next Step
The NPzT equilibration MD of the Cu(111)/water interface is now complete.
In the next notebook [03_steered_md_en.ipynb](./03_steered_md_en.ipynb), we will use PLUMED's MOVINGRESTRAINT to perform Steered MD, pulling the Cu surface atom in the z-direction.